# Demo — sample downloader / creator  ·  run on Kaggle (Internet ON)

Builds the demo samples: the **first 20** FLEURS fa_ir clips (clean) and **5 × 5 = 25** clinic-noisy clips (first 5 base clips × SNR sweep 20/15/10/5/0 dB). Outputs to `/kaggle/working/Fleurs_Clean_Samples` and `/kaggle/working/Fleurs_Noisy_Synth_Samples` — download both folders when done.

In [ ]:
!pip install -q soundfile pyroomacoustics

In [ ]:
import os, io, csv, tarfile, subprocess, tempfile
import numpy as np, soundfile as sf, requests
from scipy.signal import butter, sosfilt

REPO, LANG, SPLIT = "google/fleurs", "fa_ir", "test"
N_CLEAN, N_NOISY_BASE = 20, 5
SNR_DB = [20, 15, 10, 5, 0]
PROFILE_REF_SNR, PROFILE_NAME = 10, "clinic_realistic"
OUT_CLEAN = "/kaggle/working/Fleurs_Clean_Samples"
OUT_NOISY = "/kaggle/working/Fleurs_Noisy_Synth_Samples"

BASE    = f"https://huggingface.co/datasets/{REPO}/resolve/main/data/{LANG}"
TSV_URL = f"{BASE}/{SPLIT}.tsv"
TAR_URL = f"{BASE}/audio/{SPLIT}.tar.gz"
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
HEADERS = {"Accept-Encoding": "identity"}
if HF_TOKEN: HEADERS["Authorization"] = f"Bearer {HF_TOKEN}"
print("config ready")

In [ ]:
# ── level helpers + clinic-realistic degradation chain (matches the benchmark) ──
def _rms(x): return float(np.sqrt(np.mean(np.asarray(x, np.float64) ** 2)) + 1e-12)
def _fit(x, n):
    if len(x) == 0: return np.zeros(n, np.float32)
    if len(x) < n: x = np.tile(x, int(np.ceil(n / len(x))))
    return x[:n].astype(np.float32)

def reverb_synth(audio, sr, rng, rt60=0.4, **k):
    n = max(1, int(sr * rt60)); t = np.arange(n)
    ir = (rng.standard_normal(n) * np.exp(-6.908 * t / (rt60 * sr))).astype(np.float32); ir[0] += 1.0
    out = np.convolve(audio, ir)[:len(audio)]
    return (out * (_rms(audio) / _rms(out))).astype(np.float32)

def reverb_pyroom(audio, sr, rng, rt60=0.4, room=(4.0, 5.0, 3.0), **k):
    try:
        import pyroomacoustics as pra
        e_abs, mo = pra.inverse_sabine(rt60, list(room))
        r = pra.ShoeBox(list(room), fs=sr, materials=pra.Material(e_abs), max_order=int(mo))
        r.add_source([room[0]*0.5, room[1]*0.35, 1.2], signal=audio.astype(np.float64))
        r.add_microphone(np.array([room[0]*0.5, room[1]*0.65, 1.2]).reshape(3, 1))
        r.simulate()
        out = _fit(np.asarray(r.mic_array.signals[0], np.float32)[:len(audio)], len(audio))
        return (out * (_rms(audio) / _rms(out))).astype(np.float32)
    except Exception as e:
        print(f"    [reverb_pyroom -> synth fallback: {e}]")
        return reverb_synth(audio, sr, rng, rt60=rt60)

def add_gaussian(audio, sr, rng, snr_db=10, **k):
    p = float(np.mean(audio.astype(np.float64) ** 2)) + 1e-12
    noise = rng.normal(0.0, np.sqrt(p / (10.0 ** (snr_db / 10.0))), size=audio.shape)
    return (audio + noise.astype(np.float32)).astype(np.float32)

def bandlimit(audio, sr, rng, low=120.0, high=6000.0, order=4, **k):
    high = min(high, sr / 2 - 1)
    return sosfilt(butter(order, [low, high], btype="band", fs=sr, output="sos"), audio).astype(np.float32)

def clip_dist(audio, sr, rng, drive=0.2, **k):
    return np.clip(audio * (1.0 + drive * 6.0), -1.0, 1.0).astype(np.float32)

def codec_roundtrip(audio, sr, rng, codec="opus", bitrate="20k", **k):
    try:
        with tempfile.TemporaryDirectory() as d:
            wi, ec, wo = (os.path.join(d, f) for f in ("in.wav", "e.opus", "out.wav"))
            sf.write(wi, np.clip(audio, -1, 1).astype(np.float32), sr, subtype="PCM_16")
            subprocess.run(["ffmpeg", "-y", "-i", wi, "-c:a", "libopus", "-b:a", bitrate, ec], check=True, capture_output=True)
            subprocess.run(["ffmpeg", "-y", "-i", ec, "-ar", str(sr), "-ac", "1", wo], check=True, capture_output=True)
            y, _ = sf.read(wo, dtype="float32")
        return _fit(np.asarray(y, np.float32), len(audio))
    except Exception as e:
        print(f"    [codec skipped: {e}]"); return audio

EFFECTS = {"reverb_pyroom": (reverb_pyroom, "reverb"), "add_gaussian": (add_gaussian, "noise"),
           "bandlimit": (bandlimit, "mic"), "clip_dist": (clip_dist, "mic"), "codec_roundtrip": (codec_roundtrip, "mic")}
REALISTIC_PIPELINE = [
    ("reverb_pyroom",   dict(rt60=0.45, room=(4.0, 5.0, 3.0))),
    ("add_gaussian",    dict(snr_db=PROFILE_REF_SNR)),
    ("bandlimit",       dict(low=120.0, high=6000.0)),
    ("clip_dist",       dict(drive=0.15)),
    ("codec_roundtrip", dict(codec="opus", bitrate="20k")),
]
def apply_profile(audio, sr, snr_value, rng, ref_snr=PROFILE_REF_SNR):
    x = np.asarray(audio, np.float32).copy(); offset = snr_value - ref_snr
    for name, params in REALISTIC_PIPELINE:
        fn, cat = EFFECTS[name]; p = dict(params)
        if cat == "noise": p["snr_db"] = p.get("snr_db", ref_snr) + offset
        try: x = fn(x, sr, rng, **p)
        except Exception as e: print(f"    [warn] {name}: {e}")
    return x.astype(np.float32)

def load_transcripts():
    r = requests.get(TSV_URL, headers=HEADERS, timeout=60); r.raise_for_status()
    refs = {}
    for line in r.text.splitlines():
        c = line.split("\t")
        if len(c) >= 3: refs[c[1].strip()] = (c[3] if len(c) > 3 and c[3].strip() else c[2]).strip()
    return refs

def fetch_base_clips(n):
    refs, clips = load_transcripts(), []
    with requests.get(TAR_URL, headers=HEADERS, stream=True, timeout=300) as resp:
        resp.raise_for_status(); resp.raw.decode_content = False
        with tarfile.open(fileobj=resp.raw, mode="r|gz") as tar:
            for member in tar:
                if not (member.isfile() and member.name.endswith(".wav")): continue
                fn = os.path.basename(member.name)
                arr, sr = sf.read(io.BytesIO(tar.extractfile(member).read()), dtype="float32")
                if arr.ndim > 1: arr = arr.mean(axis=1)
                clips.append({"file": fn, "array": arr.astype(np.float32), "sr": sr, "reference": refs.get(fn, "")})
                if len(clips) >= n: break
    return clips
print("helpers ready")

In [ ]:
# ── build the samples ───────────────────────────────────────────
os.makedirs(OUT_CLEAN, exist_ok=True); os.makedirs(OUT_NOISY, exist_ok=True)
base = fetch_base_clips(N_CLEAN)
print(f"Fetched first {len(base)} FLEURS clips")

# clean: first 20
crows = []
for i, b in enumerate(base):
    fn = f"clip_{i:02d}.wav"
    sf.write(os.path.join(OUT_CLEAN, fn), b["array"], b["sr"], subtype="PCM_16")
    crows.append({"index": i, "filename": fn, "reference": b["reference"]})
with open(os.path.join(OUT_CLEAN, "metadata.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["index", "filename", "reference"]); w.writeheader(); w.writerows(crows)
print(f"clean: {len(crows)} clips -> {OUT_CLEAN}")

# noisy: first 5 base x 5 SNR = 25
rng = np.random.default_rng(1234); nrows = []; n = 0
for src_i in range(min(N_NOISY_BASE, len(base))):
    b = base[src_i]
    for snr in SNR_DB:
        noisy = apply_profile(b["array"], b["sr"], snr, rng)
        fn = f"clip_{n:02d}_snr{snr}.wav"
        sf.write(os.path.join(OUT_NOISY, fn), np.clip(noisy, -1, 1), b["sr"], subtype="PCM_16")
        nrows.append({"index": n, "filename": fn, "source_index": src_i, "snr_db": snr, "reference": b["reference"]})
        n += 1
with open(os.path.join(OUT_NOISY, "metadata.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["index", "filename", "source_index", "snr_db", "reference"]); w.writeheader(); w.writerows(nrows)
print(f"noisy: {len(nrows)} clips -> {OUT_NOISY}")
print("\nDone. Download both folders from the Kaggle output.")